In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(
    directory
    for directory in [Path.cwd(), *Path.cwd().parents]
    if (directory / "train_script.py").is_file()
)

TOP_K = 4

metric_directions = {
    "TEST_RMSE": True,
    "TEST_MAE": True,
    "TEST_R2": False,
    "TEST_BFE": True,
}

groups = [
    (
        "INITIAL EQUATIONS (OLDER EPSILON)",
        [
            (
                "template",
                "evaluated_equations_ranked_template_deriv.csv",
                3,
            ),
            (
                "unconstrained",
                "evaluated_equations_ranked_default_deriv.csv",
                None,
            ),
        ],
    ),
    (
        "NEW EQUATIONS (FIXED EPSILON)",
        [
            (
                "template",
                "epsilon_verification/template_corrected_epsilon.csv",
                1,
            ),
            (
                "unconstrained",
                "epsilon_verification/unconstrained_corrected_epsilon.csv",
                None,
            ),
        ],
    ),
]

for heading, experiments in groups:
    print(f"\n{heading}\n{'=' * len(heading)}")

    for experiment, csv_path, epsilon_position in experiments:
        equations = pd.read_csv(ROOT / csv_path)

        if epsilon_position is not None:
            injection = equations["equation"].str.split(";", n=1).str[0]
            equations["USES_EPSILON"] = injection.str.contains(
                rf"#{epsilon_position}(?!\d)",
                regex=True,
            )
        else:
            equations["USES_EPSILON"] = equations["equation"].str.contains(
                r"\bepsilon\b",
                regex=True,
            )

        print(f"\n{experiment.upper()}: {len(equations)} candidates")

        for metric, ascending in metric_directions.items():
            ranked = equations.sort_values(
                metric,
                ascending=ascending,
                kind="stable",
            ).reset_index(drop=True)

            print(f"\n{experiment.upper()} — {metric}")

            with pd.option_context("display.max_colwidth", None):
                display(
                    ranked.head(TOP_K)[
                        ["equation", "complexity", metric, "USES_EPSILON"]
                    ]
                )


INITIAL EQUATIONS (OLDER EPSILON)

TEMPLATE: 393 candidates

TEMPLATE — TEST_RMSE


,equation,complexity,TEST_RMSE,USES_EPSILON
0,g = (#2 * -0.0010713526) * sqrt(#1 + 1.32442); d = square((#1 * 0.01838707) - 0.5912093),14,13.806113,False
1,g = #2 * (sqrt(#1 - -1.2173425) * -0.0010723427); d = square((#1 * -0.018120795) + 0.5971914),14,13.888641,False
2,g = (sqrt(#1 - -1.1051167) * #2) * -0.0010888062; d = square((#1 * -0.018209398) + 0.59240085),14,13.912392,False
3,g = #2 * (sqrt(#1 - -1.1162246) * -0.0010842164); d = square((#1 * -0.01812232) + 0.5913376),14,13.917891,False



TEMPLATE — TEST_MAE


,equation,complexity,TEST_MAE,USES_EPSILON
0,g = (#2 * -0.0010713526) * sqrt(#1 + 1.32442); d = square((#1 * 0.01838707) - 0.5912093),14,10.562884,False
1,g = #2 * (sqrt(#1 - -1.1162246) * -0.0010842164); d = square((#1 * -0.01812232) + 0.5913376),14,10.589469,False
2,g = #2 * (sqrt(#1 - -1.2173425) * -0.0010723427); d = square((#1 * -0.018120795) + 0.5971914),14,10.594898,False
3,g = (sqrt(#1 - -1.1051167) * #2) * -0.0010888062; d = square((#1 * -0.018209398) + 0.59240085),14,10.597599,False



TEMPLATE — TEST_R2


,equation,complexity,TEST_R2,USES_EPSILON
0,g = (#2 * -0.0010713526) * sqrt(#1 + 1.32442); d = square((#1 * 0.01838707) - 0.5912093),14,0.789847,False
1,g = #2 * (sqrt(#1 - -1.1162246) * -0.0010842164); d = square((#1 * -0.01812232) + 0.5913376),14,0.788447,False
2,g = #2 * (sqrt(#1 - -1.2173425) * -0.0010723427); d = square((#1 * -0.018120795) + 0.5971914),14,0.788438,False
3,g = (sqrt(#1 - -1.1051167) * #2) * -0.0010888062; d = square((#1 * -0.018209398) + 0.59240085),14,0.788294,False


... (22 outputs left out)


UNCONSTRAINED — TEST_R2


,equation,complexity,TEST_R2,USES_EPSILON
0,(square(-17.15375 - (DST * -0.5683326)) - (VBs * sqrt(P_dyn - -1.2791218))) * 0.0010682555,15,0.788812,False
1,((VBs * sqrt(P_dyn + 1.1150439)) - square((DST * -0.59780794) - -15.503676)) / (DST + -836.2796),17,0.787940,False
2,square((DST * -0.018247437) - -0.5944656) + (sqrt(P_dyn - -1.2362556) * (VBs * -0.0010782854)),15,0.787734,False
3,square((DST * -0.018144378) + 0.59190536) + ((VBs * -0.001057952) * sqrt(P_dyn + 1.2858895)),15,0.787665,False



UNCONSTRAINED — TEST_BFE


,equation,complexity,TEST_BFE,USES_EPSILON
0,((VBs * sqrt(P_dyn + 1.1150439)) - square((DST * -0.59780794) - -15.503676)) / (DST + -836.2796),17,15.846162,False
1,((VBs * ((P_dyn - (DST * -0.020267786)) + 12.099897)) * -0.00015967918) + (-0.7036516 - (square(DST + -59.507454) * -0.00026096366)),20,15.976549,False
2,((VBs * sqrt(P_dyn + 1.2546012)) - square((P_dyn + 14.7217655) + (DST * -0.5883604))) / (DST + (DST + -791.372)),21,16.171727,False
3,(((DST - ((VBs * 0.03565441) + 287.94803)) * DST) - ((P_dyn + 14.21617) * VBs)) / 6529.3853,17,16.251270,False
